### ColumnTransformer:
* ColumnTransformer allows us to apply encoders at once with worrying about shifting of columns after encoding/transformation.
* It's like the `apply` method of `Pandas` but on steroids.

In [19]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

#import the Column Transformer
from sklearn.compose import ColumnTransformer

#Import the encoders
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

In [20]:
US_housing_df = pd.read_csv(filepath_or_buffer = '~/ML/Datasets/US-Housing-Price-Index.csv')

In [21]:
US_housing_df.sample(n = 6)

,hpi_type,hpi_flavor,frequency,level,place_name,place_id,yr,period,index_nsa,index_sa
111964,traditional,purchase-only,quarterly,MSA,"St. Petersburg-Clearwater-Largo, FL (MSAD)",41304,2024,3,707.01,698.35
91853,traditional,expanded-data,quarterly,MSA,"Virginia Beach-Chesapeake-Norfolk, VA-NC",47260,1997,2,111.51,111.20
48061,traditional,all-transactions,quarterly,MSA,"New Haven, CT",35300,2018,1,184.97,NaN
62081,traditional,all-transactions,quarterly,MSA,"Sheboygan, WI",43100,2011,1,156.26,NaN
95307,traditional,expanded-data,quarterly,State,Michigan,MI,2004,3,208.71,206.23
127648,non-metro,all-transactions,quarterly,State,Virginia,VA,2003,3,143.78,NaN


In [22]:
US_housing_df['level'].unique()

<StringArray>
['USA or Census Division', 'MSA', 'State', 'Puerto Rico']
Length: 4, dtype: str

In [23]:
US_housing_df.shape

(130697, 10)

In [24]:
US_housing_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 130697 entries, 0 to 130696
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   hpi_type    130697 non-null  str    
 1   hpi_flavor  130697 non-null  str    
 2   frequency   130697 non-null  str    
 3   level       130697 non-null  str    
 4   place_name  130697 non-null  str    
 5   place_id    130697 non-null  str    
 6   yr          130697 non-null  int64  
 7   period      130697 non-null  int64  
 8   index_nsa   130697 non-null  float64
 9   index_sa    43429 non-null   float64
dtypes: float64(2), int64(2), str(6)
memory usage: 10.0 MB


The columns 'hpi_type', 'hpi_flavor', 'place_name' are nominal and 'level', 'frequency' are ordinal category data.
So we need to apply
* One hot encoding to 'hpi_type', 'hpi_flavor' and 'place_name'
* Ordinal encoding to 'level' and 'frequency'.


In [25]:
US_housing_df.columns

Index(['hpi_type', 'hpi_flavor', 'frequency', 'level', 'place_name',
       'place_id', 'yr', 'period', 'index_nsa', 'index_sa'],
      dtype='str')

In [35]:
US_housing_df.drop(columns = ['place_id', 'index_sa', 'index_nsa'])

,hpi_type,hpi_flavor,frequency,level,place_name,yr,period
0,traditional,purchase-only,monthly,USA or Census Division,East North Central Division,1991,1
1,traditional,purchase-only,monthly,USA or Census Division,East North Central Division,1991,2
2,traditional,purchase-only,monthly,USA or Census Division,East North Central Division,1991,3
3,traditional,purchase-only,monthly,USA or Census Division,East North Central Division,1991,4
4,traditional,purchase-only,monthly,USA or Census Division,East North Central Division,1991,5
...,...,...,...,...,...,...,...
130692,developmental,purchase-only,quarterly,Puerto Rico,Puerto Rico,2024,1
130693,developmental,purchase-only,quarterly,Puerto Rico,Puerto Rico,2024,2
130694,developmental,purchase-only,quarterly,Puerto Rico,Puerto Rico,2024,3
130695,developmental,purchase-only,quarterly,Puerto Rico,Puerto Rico,2024,4


* The transformations in a ColumnTransformer work separately/independently of each other and only merge/concatenate in the final product
* Example: ColumnTransform is defined with a OrdinalEncoder(on features, 'A', 'B') and a OneHotEncoder (On Features, 'C', 'D') and is fit_transform to X_train, OrdinalEncoder and OneHotEncoder are given their respective features and when the appropriate transformation is done. The results and the remaining columns are concatenated in order and returned as a NumPy array.

In [ ]:
transformer = ColumnTransformer(transformers = [
    ('tnf1', OrdinalEncoder(categories = [['monthly', 'quarterly']]), ['frequency']),
    ('tnf2', OneHotEncoder(drop = 'first'), ['hpi_type', 'hpi_flavor', 'level'])
], remainder = 'passthrough')

transformer.fit_transform(US_housing_df)

array([[0.0, 0.0, 0.0, ..., 1, 100.0, 100.0],
       [0.0, 0.0, 0.0, ..., 2, 100.87, 100.87],
       [0.0, 0.0, 0.0, ..., 3, 101.32, 100.9],
       ...,
       [1.0, 0.0, 0.0, ..., 3, 233.53, 226.83],
       [1.0, 0.0, 0.0, ..., 4, 229.51, 236.17],
       [1.0, 0.0, 0.0, ..., 1, 245.52, 248.97]],
      shape=(130697, 16), dtype=object)

In [ ]:
US_housing_df = pd.DataFrame(transformer.fit_transform(US_housing_df), columns = transformer.get_feature_names_out(), index = US_housing_df.index) # type: ignore

US_housing_df.sample(n = 5)

,tnf1__frequency,tnf2__hpi_type_distress-free,tnf2__hpi_type_manufactured,tnf2__hpi_type_non-metro,tnf2__hpi_type_traditional,tnf2__hpi_flavor_expanded-data,tnf2__hpi_flavor_purchase-only,tnf2__level_Puerto Rico,tnf2__level_State,tnf2__level_USA or Census Division,remainder__place_name,remainder__place_id,remainder__yr,remainder__period,remainder__index_nsa,remainder__index_sa
62871,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,"Sioux Falls, SD-MN",43620,2019,3,231.13,NaN
95082,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,Maryland,MD,2016,4,198.62,198.2
93771,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,Hawaii,HI,1997,2,123.02,121.95
65908,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,"Texarkana, TX-AR",45500,2024,4,299.99,NaN
116359,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,Indiana,IN,1993,1,107.0,107.02


### NOTE:

* Just beacuse one ColumnTransformer is not effected by the columns shiftting and renaming, does not mean that you should chain the ColumnTransformers because now they will work sequentially meaning the one ColumnTransformer will recive a NumPy array which has been already processed by another ColumnTransformer.
* This means that you would have to manually keep track of indices or the names of the columns after each ColumnTransformer.
* Good news is that Scikit Learn's Pipelines simplify this task.